### prepair modules and bases settings

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix,  classification_report, log_loss
from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm
import scipy.io
import re
import itertools

import os
from os.path import join
import contextlib
from copy import deepcopy
import imp 
import time 
import sys

import pickle
from pdb import set_trace

from IPython.display import clear_output, display

In [8]:
# Add the directory containing your modules to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'ses2_modelstims')))

# load local functions
import stim_io
import stim_io_plotting
import vtc
import bvbabel

In [9]:
import regression
import varpar

In [10]:
## LOADING GRID

# Load from MAT file
variables = scipy.io.loadmat('/media/jorvhar/Data8T/MRIData/timing data/grid_parameters_python.mat')

# Extract individual variables
tunsteps = variables['tunsteps']
freqstep = variables['freqstep']
subsample = variables['subsample']
mustep = variables['mustep']
muarray_bins = variables['muarray_bins']
muarray = variables['muarray']
fwhm = variables['fwhm']
octgrid = variables['octgrid']
sigmagrid = variables['sigmagrid']
pref_range = variables['pref_range']
sharp_range_fwhm = variables['sharp_range_fwhm']
sharp_range = variables['sharp_range']

## 1. Set up regresiion model
Options:

In [11]:
### --- REGRESSION SAVING OPTIONS ---

# set modeltype
# modeltype = LinearRegression() #can be LinearRegression (ols), Ridge(alpha=..), Lasso(alpha=..)  etc.
modeltype = LinearRegression() 
key_ai = ['raw_scores', 'coefs', 'intercepts', 'correlation'] # what keys to median and mean across folds

# model return options
save_predict = False          # save y_pred-y
score_of_interest = 'score'   # what score to use  'score', 'raw_scores', 'coefs', 'intercepts', 'correlation'

# outlier options - #tobeimplemented
SD_lim = 3                    # remove y x sd higher then mean
remove_outliers = False       # if false dont remove sd outliers 

# what regressor variant to use
convolved = True   # use convolved dataset
resampled = True   # use scipy resampled data, instead of standard mean for downsampled data

zs=False   # zscore y
ts=True   # temporally smooth y - desired, if not too broad - idealy matching HRF (2.8/1.8=1.56)
hp=False  # highpass filter y - not wanted

# add drift regressors
dr=False   # drift regressor
br=True

### --- LOCATION OPTIONS ---

# file location
mridat_dir = '/media/jorvhar/Data8T/MRIData/PreProc'
logdat_dir = '/media/jorvhar/Data8T/MRIData/timing data/data'
vtc_dir = '/media/jorvhar/New Volume1/vtcs' # adviced to put vtc's on a (nvme) ssd while running analyses
pp_dir = lambda pp, ses : f'S{pp:02d}_SES{ses}'
betas_dir = 'Betas'

# tonotopy and mask filenames
tonotopy_vmp = 'prf_permutations_for_s2.vmp'
mask_fn = 'gm-subcortical.msk'

# fn lambda
fn = lambda pp, ses, run : f'S{pp:02d}_SES{ses}_run{run}_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc'


### --- PARTICIPANT OPTIONS ---

# session of interest
ses = 2

# variable that may be different per participant
ppz = [1,2,3,4,5,6,7,8,9,10]
# n_splitsz = [6,6,5,5,5,5,5,5,5,5]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_splitsz = [12,12,10,10,10,10,10,10,10,10]
n_runz = [12,12,10,10,10,10,10,10,10,10]        # number of runs
startpp = 1

### --- SET THEORIE REGRESSION MODELS ---

# for full 3 sets we need 7 sets
models = ['base_U_adaptation_U_prediction']
# if we want to use sets we only need 3 models 
##models=['base_U_adaptation', 'prediction', 'base_U_adaptation_U_prediction']

## REGRESSORS IN MODELS ##
model_regressors = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'error',       # voxelwise error
                                   'surprisal',   # global prior surprise
                                   'prec_w_surprisal',   # global prior surprise
                                   'precision'  # global precision
                                  ]  
                   } 
# if we want to add adapted activation
# model_regressors['adaptation'] += ['adapt_activ']

# set combination of regressors
model_regressors.update({'base_U_adaptation':             model_regressors['base']+
                                                          model_regressors['adaptation'],
                        'base_U_prediction':              model_regressors['base']+
                                                          model_regressors['prediction'], 
                        'adaptation_U_prediction':        model_regressors['adaptation']+
                                                          model_regressors['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors['base']+
                                                          model_regressors['adaptation']+
                                                          model_regressors['prediction']})
y_var = 'voxeltimecourse'

# select fn
pick_fn_prefix = 'ANTS_scores_blocked_effects'


## NOTE FOR AFTER VACATION
## 1. ADD PRECISION TO PREDICTION MODEL X
## 2. ADD PER BLOCK DRIFT REGRESSORS, GRADIENTS (LINSPACE) FROM 0-1
## 3. ADD PRECISION WEIGHTED PRED ERROR?
## 4. ADD THE MOTION PARAMETERS

## COMPARE HRF SLOWER AND FASTER, FOR PP2

## 2. Run regressions - per participant - per model - per gridpostion 
Run the full regressions, looping over participants, copy pasting files to a suitable ssd location, and doing the regression for the full grid.

In [ ]:
# loop over all participants
for pp_idx in np.arange(ppz.index(startpp),len(ppz)):

    ### --- PREPARE PARTICIPANT DATA ---

    # fetch current pp vars
    pp = ppz[pp_idx]
    runz = np.arange(1,n_runz[pp_idx]+1)
    n_splits = n_splitsz[pp_idx]

    print(F'--RUNNING REGRESSION LOOP FOR PP: {pp} (runs={n_runz[pp_idx]},nr_splits={n_splits})--')

    # load stim df and tr df
    stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_v2')
    tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_v2')

    # create full path for vmp and mask
    mskpath = join(mridat_dir, pp_dir(pp,1), mask_fn)
    vmppath = join(mridat_dir, pp_dir(pp,1), betas_dir, tonotopy_vmp)

    # load full mask and convert to indeces
    _, msk = bvbabel.msk.read_msk(mskpath)
    msk = np.where(msk)

    # load vmp image
    vmp_head, vmp_img = bvbabel.vmp.read_vmp(vmppath)

    # load list of filenames at origin, and vtc filenames
    origin_fns = [join(mridat_dir, pp_dir(pp, ses), fn(pp,ses,run)) for run in runz]
    vtc_fns = [join(vtc_dir, fn(pp,ses,run)) for run in runz]

    # copy files to ssd for efficient and fast chuck processing
    stim_io.copy_files(origin_fns, vtc_fns)

    # load tonotopy vmp
    vmp_df = stim_io.vmp_add_realsigma(vmp_img, msk, mustep[0][0]) # 1. prfMU, 2. prfMU_hz, prfS, prfO


    ### --- RUN FULL REGRESSION ---

    # run full regression for current pp
    scores = regression.run_model_grid(tr_df,stim_df,vmp_df,vtc_fns,
                                       msk, vmp_img,
                                       pref_range,sharp_range,
                                       models, model_regressors,
                                       mustep, n_splits, modeltype, key_ai,
                                       save_predict=save_predict, 
                                       convolved=convolved, resampled=resampled,
                                       zs=zs, ts=ts, hp=hp, dr=dr, br=br)
    # clean up prints - for next pp
    clear_output(wait=True)

    # save scores
    if not os.path.exists(join(mridat_dir, pp_dir(pp, ses), 'Betas')):
        os.mkdir(join(mridat_dir, pp_dir(pp, ses), 'Betas'))

    # append the pickle result naming based on cleaning steps 
    pick_fn = pick_fn_prefix #'scores_prec'
    if ts: pick_fn = f'{pick_fn}_tempsmooth'
    if hp: pick_fn = f'{pick_fn}_highpass'
    if dr: pick_fn = f'{pick_fn}_drift'
    # pickle the results
    with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'wb') as handle:
        pickle.dump(scores, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # loading of pickled results
    ###with open(join(mridat_dir, pp_dir(pp, ses), 'Betas/scores.pickle'), 'rb') as handle:
    ###    scores = pickle.load(handle)

    # clean up files where needed for next pp
    for fp in vtc_fns:
        os.remove(fp)



--RUNNING REGRESSION LOOP FOR PP: 2 (runs=12,nr_splits=12)--
Copied /media/jorvhar/Data8T/MRIData/PreProc/S02_SES2/S02_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S02_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S02_SES2/S02_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S02_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S02_SES2/S02_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S02_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S02_SES2/S02_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S02_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S02_SES2/S02_SES2_run5_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Vol

grid: 49/2400, 
        -current chuck took: 1.14 seconds
        -estimated time elapsed: 1.68 minutes of 82.39 minutes
grid: 50/2400, 
        -current chuck took: 2.65 seconds
        -estimated time elapsed: 1.73 minutes of 82.86 minutes
grid: 51/2400, 
        -current chuck took: 1.75 seconds
        -estimated time elapsed: 1.76 minutes of 82.61 minutes
grid: 52/2400, 
        -current chuck took: 0.66 seconds
        -estimated time elapsed: 1.77 minutes of 81.53 minutes
grid: 53/2400, 
        -current chuck took: 1.06 seconds
        -estimated time elapsed: 1.78 minutes of 80.79 minutes
grid: 54/2400, 
        -current chuck took: 0.86 seconds
        -estimated time elapsed: 1.80 minutes of 79.93 minutes
grid: 55/2400, 
        -current chuck took: 2.63 seconds
        -estimated time elapsed: 1.84 minutes of 80.39 minutes
grid: 56/2400, 
        -current chuck took: 1.83 seconds
        -estimated time elapsed: 1.87 minutes of 80.27 minutes
grid: 57/2400, 
        -current

grid: 117/2400, 
        -current chuck took: 1.96 seconds
        -estimated time elapsed: 3.41 minutes of 69.98 minutes
grid: 118/2400, 
        -current chuck took: 1.17 seconds
        -estimated time elapsed: 3.43 minutes of 69.78 minutes
grid: 119/2400, 
        -current chuck took: 1.50 seconds
        -estimated time elapsed: 3.46 minutes of 69.70 minutes
grid: 120/2400, 
        -current chuck took: 1.28 seconds
        -estimated time elapsed: 3.48 minutes of 69.54 minutes
grid: 121/2400, 
        -current chuck took: 0.31 seconds
        -estimated time elapsed: 3.48 minutes of 69.07 minutes
grid: 122/2400, 
        -current chuck took: 0.29 seconds
        -estimated time elapsed: 3.49 minutes of 68.60 minutes
grid: 123/2400, 
        -current chuck took: 0.70 seconds
        -estimated time elapsed: 3.50 minutes of 68.27 minutes
grid: 124/2400, 
        -current chuck took: 0.83 seconds
        -estimated time elapsed: 3.51 minutes of 67.99 minutes
grid: 125/2400, 
       

grid: 185/2400, 
        -current chuck took: 0.58 seconds
        -estimated time elapsed: 4.74 minutes of 61.51 minutes
grid: 186/2400, 
        -current chuck took: 2.42 seconds
        -estimated time elapsed: 4.78 minutes of 61.70 minutes
grid: 187/2400, 
        -current chuck took: 0.26 seconds
        -estimated time elapsed: 4.79 minutes of 61.43 minutes
grid: 188/2400, 
        -current chuck took: 1.05 seconds
        -estimated time elapsed: 4.80 minutes of 61.32 minutes
grid: 189/2400, 
        -current chuck took: 1.89 seconds
        -estimated time elapsed: 4.84 minutes of 61.40 minutes
grid: 190/2400, 
        -current chuck took: 2.50 seconds
        -estimated time elapsed: 4.88 minutes of 61.60 minutes
grid: 191/2400, 
        -current chuck took: 0.61 seconds
        -estimated time elapsed: 4.89 minutes of 61.41 minutes
grid: 192/2400, 
        -current chuck took: 0.60 seconds
        -estimated time elapsed: 4.90 minutes of 61.21 minutes
grid: 193/2400, 
       

grid: 253/2400, 
        -current chuck took: 0.28 seconds
        -estimated time elapsed: 6.10 minutes of 57.89 minutes
grid: 254/2400, 
        -current chuck took: 0.31 seconds
        -estimated time elapsed: 6.11 minutes of 57.71 minutes
grid: 255/2400, 
        -current chuck took: 0.62 seconds
        -estimated time elapsed: 6.12 minutes of 57.58 minutes
grid: 256/2400, 
        -current chuck took: 0.87 seconds
        -estimated time elapsed: 6.13 minutes of 57.49 minutes
grid: 257/2400, 
        -current chuck took: 1.27 seconds
        -estimated time elapsed: 6.15 minutes of 57.47 minutes
grid: 258/2400, 
        -current chuck took: 0.55 seconds
        -estimated time elapsed: 6.16 minutes of 57.33 minutes
grid: 259/2400, 
        -current chuck took: 4.49 seconds
        -estimated time elapsed: 6.24 minutes of 57.80 minutes
grid: 260/2400, 
        -current chuck took: 1.41 seconds
        -estimated time elapsed: 6.26 minutes of 57.79 minutes
grid: 261/2400, 
       

...

grid: 862/2400, 
        -current chuck took: 1.86 seconds
        -estimated time elapsed: 24.29 minutes of 67.62 minutes
grid: 863/2400, 
        -current chuck took: 1.47 seconds
        -estimated time elapsed: 24.31 minutes of 67.61 minutes
grid: 864/2400, 
        -current chuck took: 1.11 seconds
        -estimated time elapsed: 24.33 minutes of 67.59 minutes
grid: 865/2400, 
        -current chuck took: 2.09 seconds
        -estimated time elapsed: 24.37 minutes of 67.61 minutes
grid: 866/2400, 
        -current chuck took: 1.03 seconds
        -estimated time elapsed: 24.38 minutes of 67.58 minutes
grid: 867/2400, 
        -current chuck took: 2.94 seconds
        -estimated time elapsed: 24.43 minutes of 67.63 minutes
grid: 868/2400, 
        -current chuck took: 3.16 seconds
        -estimated time elapsed: 24.48 minutes of 67.70 minutes
grid: 869/2400, 
        -current chuck took: 1.99 seconds
        -estimated time elapsed: 24.52 minutes of 67.71 minutes
grid: 870/2400, 

grid: 929/2400, 
        -current chuck took: 3.06 seconds
        -estimated time elapsed: 26.80 minutes of 69.23 minutes
grid: 930/2400, 
        -current chuck took: 2.88 seconds
        -estimated time elapsed: 26.85 minutes of 69.28 minutes
grid: 931/2400, 
        -current chuck took: 1.59 seconds
        -estimated time elapsed: 26.87 minutes of 69.28 minutes
grid: 932/2400, 
        -current chuck took: 1.81 seconds
        -estimated time elapsed: 26.90 minutes of 69.28 minutes
grid: 933/2400, 
        -current chuck took: 1.40 seconds
        -estimated time elapsed: 26.93 minutes of 69.26 minutes
grid: 934/2400, 
        -current chuck took: 1.19 seconds
        -estimated time elapsed: 26.95 minutes of 69.24 minutes
grid: 935/2400, 
        -current chuck took: 2.17 seconds
        -estimated time elapsed: 26.98 minutes of 69.26 minutes
grid: 936/2400, 
        -current chuck took: 1.80 seconds
        -estimated time elapsed: 27.01 minutes of 69.26 minutes
grid: 937/2400, 

grid: 996/2400, 
        -current chuck took: 1.60 seconds
        -estimated time elapsed: 29.12 minutes of 70.16 minutes
grid: 997/2400, 
        -current chuck took: 1.72 seconds
        -estimated time elapsed: 29.15 minutes of 70.16 minutes
grid: 998/2400, 
        -current chuck took: 2.76 seconds
        -estimated time elapsed: 29.19 minutes of 70.20 minutes
grid: 999/2400, 
        -current chuck took: 3.12 seconds
        -estimated time elapsed: 29.24 minutes of 70.25 minutes
grid: 1000/2400, 
        -current chuck took: 3.03 seconds
        -estimated time elapsed: 29.29 minutes of 70.30 minutes
grid: 1001/2400, 
        -current chuck took: 1.55 seconds
        -estimated time elapsed: 29.32 minutes of 70.30 minutes
grid: 1002/2400, 
        -current chuck took: 2.10 seconds
        -estimated time elapsed: 29.35 minutes of 70.31 minutes
grid: 1003/2400, 
        -current chuck took: 2.77 seconds
        -estimated time elapsed: 29.40 minutes of 70.35 minutes
grid: 1004/2

grid: 1065/2400, 
        -current chuck took: 1.49 seconds
        -estimated time elapsed: 31.62 minutes of 71.26 minutes
grid: 1066/2400, 
        -current chuck took: 1.60 seconds
        -estimated time elapsed: 31.65 minutes of 71.25 minutes
grid: 1067/2400, 
        -current chuck took: 2.61 seconds
        -estimated time elapsed: 31.69 minutes of 71.28 minutes
grid: 1068/2400, 
        -current chuck took: 1.92 seconds
        -estimated time elapsed: 31.72 minutes of 71.29 minutes
grid: 1069/2400, 
        -current chuck took: 3.00 seconds
        -estimated time elapsed: 31.77 minutes of 71.33 minutes
grid: 1070/2400, 
        -current chuck took: 2.46 seconds
        -estimated time elapsed: 31.81 minutes of 71.36 minutes
grid: 1071/2400, 
        -current chuck took: 1.08 seconds
        -estimated time elapsed: 31.83 minutes of 71.33 minutes
grid: 1072/2400, 
        -current chuck took: 1.70 seconds
        -estimated time elapsed: 31.86 minutes of 71.33 minutes
grid: 10

grid: 1132/2400, 
        -current chuck took: 1.67 seconds
        -estimated time elapsed: 34.12 minutes of 72.34 minutes
grid: 1133/2400, 
        -current chuck took: 2.33 seconds
        -estimated time elapsed: 34.16 minutes of 72.36 minutes
grid: 1134/2400, 
        -current chuck took: 2.19 seconds
        -estimated time elapsed: 34.20 minutes of 72.38 minutes
grid: 1135/2400, 
        -current chuck took: 1.48 seconds
        -estimated time elapsed: 34.22 minutes of 72.36 minutes
grid: 1136/2400, 
        -current chuck took: 3.58 seconds
        -estimated time elapsed: 34.28 minutes of 72.43 minutes
grid: 1137/2400, 
        -current chuck took: 3.59 seconds
        -estimated time elapsed: 34.34 minutes of 72.49 minutes
grid: 1138/2400, 
        -current chuck took: 2.57 seconds
        -estimated time elapsed: 34.38 minutes of 72.52 minutes
grid: 1139/2400, 
        -current chuck took: 3.26 seconds
        -estimated time elapsed: 34.44 minutes of 72.57 minutes
grid: 11